In [ ]:
# In this file, we compare the accuracy and types of testbench for Llama, Qwen and commercial models.

In [9]:
import json
from collections import defaultdict

# Load type by line number (1-indexed)
data_type = {}
with open("testbench_type&score.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        obj = json.loads(line)
        data_type[i] = obj["type"]  

# Load Llama outcomes; ID field corresponds to line number 1..1000
records = []
with open("SFT/Llama3.2-1B/1000QTSA_test/Outcome_1000.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        records.append(obj)

# Identify model columns (exclude ID and Correct Answer)
sample = records[0]
model_cols = [k for k in sample.keys() if k not in ("ID", "Correct Answer")]

# Compute accuracy per model per type
acc_by_type = {m: defaultdict(lambda: [0, 0]) for m in model_cols}

for rec in records:
    rid = int(rec["ID"])
    qtype = data_type.get(rid)  
    if qtype is None:
        continue
    correct = rec["Correct Answer"]
    for m in model_cols:
        ans = rec.get(m)
        if ans is None:
            continue
        acc_by_type[m][qtype][1] += 1
        if ans == correct:
            acc_by_type[m][qtype][0] += 1

# Output the table
print("=" * 100)
print("Accuracy by question type (excluding null answers)")
print("=" * 100)

all_types = sorted(set(t for d in acc_by_type.values() for t in d.keys()))
print(f"{'Model':<50} " + " ".join(f"{t}" for t in all_types))
print("-" * 100)

for m in model_cols:
    parts = []
    for t in all_types:
        c, total = acc_by_type[m][t]
        if total == 0:
            parts.append("  N/A")
        else:
            parts.append(f"{c/total*100:5.1f}")
    print(f"{m:<50} " + " ".join(parts))

Accuracy by question type (excluding null answers)
Model                                              Application & Limitations Conceptual Understanding Design Implementation Methematical Derivation Performance Analysis
----------------------------------------------------------------------------------------------------
1B_prompt2 Answer                                   15.3  18.0  24.9  18.4  21.9
1B_Instruct_prompt2 Answer                          41.5  37.2  41.5  43.9  40.9
3B_Instruct_prompt2 Answer                          69.4  73.1  73.4  56.5  64.3
3B_prompt2 Answer                                   68.9  65.2  75.3  76.1  72.3
finetuned_1B_prompt2 Answer                         54.4  54.4  56.8  45.5  51.4
finetuned_1B_Instruct_prompt2 Answer                66.0  64.9  63.4  54.5  60.3
finetuned_3B_Instruct_prompt2 Answer                81.1  79.4  77.7  62.8  74.4
finetuned_3B_prompt2 Answer                         67.0  73.6  71.8  50.0  64.1
(new)1B_prompt2 Answer         

In [12]:
import json
from collections import defaultdict

# Load question types by line number (1-indexed)
question_type = {}
with open("testbench_type&score.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        obj = json.loads(line)
        question_type[i] = obj["type"] 

# Load Llama outcomes; ID field corresponds to line number 1..1000
records = []
with open("SFT/Qwen3-0.6B/1000QTSA_test/Outcome_1000.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        records.append(obj)

# Identify model columns (exclude ID and Correct Answer)
sample = records[0]
model_cols = [k for k in sample.keys() if k not in ("ID", "Correct Answer")]

# Compute accuracy per model per question type
acc_by_type = {m: defaultdict(lambda: [0, 0]) for m in model_cols}

for rec in records:
    rid = int(rec["ID"])
    qtype = question_type.get(rid)  
    if qtype is None:
        continue
    correct = rec["Correct Answer"]
    for m in model_cols:
        ans = rec.get(m)
        if ans is None:
            continue
        acc_by_type[m][qtype][1] += 1
        if ans == correct:
            acc_by_type[m][qtype][0] += 1

# Get all unique question types and sort them
all_types = sorted(set(t for d in acc_by_type.values() for t in d.keys()))

# Output the table
print("=" * 100)
print("Accuracy by question type (excluding null answers)")
print("=" * 100)
print(f"{'Model':<50} " + " ".join(f"{t:>6}" for t in all_types))
print("-" * 100)

for m in model_cols:
    parts = []
    for t in all_types:
        c, total = acc_by_type[m][t]
        if total == 0:
            parts.append("  N/A")
        else:
            parts.append(f"{c/total*100:5.1f}")
    print(f"{m:<50} " + " ".join(parts))

Accuracy by question type (excluding null answers)
Model                                              Application & Limitations Conceptual Understanding Design Implementation Methematical Derivation Performance Analysis
----------------------------------------------------------------------------------------------------
nothinking_0.6B_prompt2 Answer                      67.5  65.0  68.1  51.8  61.6
nothinking_1.7B_prompt2 Answer                      74.7  68.5  69.1  55.5  64.5
nothinking_4B_prompt2 Answer                        83.0  81.6  83.1  68.1  75.4
1.7B_prompt2 Answer                                 75.8  72.6  73.4  59.7  69.7
0.6B_prompt2 Answer                                 67.0  62.9  68.6  57.1  62.1
4B_prompt2 Answer                                   86.6  85.3  83.6  75.9  81.5
nothinking_1.7B_prompt2_finetuned Answer            77.6  76.4  72.7  63.8  74.5
nothinking_0.6B_prompt2_finetuned Answer            75.8  74.2  70.6  52.1  68.1
thinking_1.7B_prompt2_finetuned

In [14]:
import json
from collections import defaultdict

# Load question types by line number (1-indexed)
question_type = {}
with open("testbench_type&score.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        obj = json.loads(line)
        question_type[i] = obj["type"] 

files = [
    ("DeepSeek", "/RAG/deepseek_testing/results/Outcome_1000.jsonl"),
    ("GPT", "/RAG/gpt_testing/results/Outcome_1000.jsonl"),
]

for source, path in files:
    print("=" * 90)
    print(f"{source}: ")
    print("=" * 90)

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            records.append(obj)

    model_cols = [k for k in records[0].keys() if k not in ("ID", "Correct Answer")]

    acc_by_type = {m: defaultdict(lambda: [0, 0]) for m in model_cols}
    overall_by_type = defaultdict(lambda: [0, 0])

    for rec in records:
        rid = int(rec["ID"])
        qtype = question_type.get(rid)  
        if qtype is None:
            continue
        correct = rec["Correct Answer"]
        for m in model_cols:
            ans = rec.get(m)
            acc_by_type[m][qtype][1] += 1
            if ans == correct:
                acc_by_type[m][qtype][0] += 1
            overall_by_type[qtype][1] += 1
            if ans == correct:
                overall_by_type[qtype][0] += 1

    # Get all unique question types and sort them
    all_types = sorted(set(t for d in acc_by_type.values() for t in d.keys()))
    
    print(f"{'Model':<55} " + " ".join(f"{t:>10}" for t in all_types) + "   Overall")
    print("-" * (55 + 11 * len(all_types) + 10))
    
    for m in model_cols:
        accs = []
        total_c, total_t = 0, 0
        for t in all_types:
            c, total = acc_by_type[m][t]
            if total > 0:
                acc = c / total * 100
                accs.append(f"{acc:5.1f}")
                total_c += c
                total_t += total
            else:
                accs.append("  N/A")
        overall = total_c / total_t * 100 if total_t > 0 else 0
        acc_str = " ".join(accs)
        print(f"{m:<55} {acc_str}    {overall:5.1f}")

    print(f"\n  Pooled accuracy by question type (all variants together):")
    for t in all_types:
        c, total = overall_by_type[t]
        if total > 0:
            print(f"    {t}: {c/total*100:5.1f}% ({c}/{total})")
        else:
            print(f"    {t}: N/A (0/{total})")
    print()

DeepSeek: 
Model                                                   Application & Limitations Conceptual Understanding Design Implementation Methematical Derivation Performance Analysis   Overall
------------------------------------------------------------------------------------------------------------------------
Deepseek-V3.2-Thinking Answer                            90.7  92.4  88.9  83.2  89.6     89.0
Deepseek-V3.2-Thinking with Basic RAG Answer             99.0  95.9  94.2  87.4  91.9     93.7
Deepseek-V3.2-Thinking with Hybrid RAG Answer            94.3  93.4  93.7  85.9  90.5     91.6

  Pooled accuracy by question type (all variants together):
    Application & Limitations:  94.7% (551/582)
    Conceptual Understanding:  93.9% (555/591)
    Design Implementation:  92.3% (573/621)
    Methematical Derivation:  85.5% (490/573)
    Performance Analysis:  90.7% (574/633)

GPT: 
Model                                                   Application & Limitations Conceptual Understand